In [ ]:
import pandas as pd
import numpy as np

# select significantly reversed miRNA pairs
f = open('kazuki_data.json')
kazuki_data = pd.read_json(f, orient='split')
f.close()
f = open('watara_data.json')
watara_data = pd.read_json(f, orient='split')
f.close()
f = open('yokoi_data.json')
yokoi_data = pd.read_json(f, orient='split')
f.close()

#non cancer parts
non_cancer_data = pd.concat([watara_data.loc[0,:],yokoi_data.loc[0,:]])
cancer_data = pd.concat([watara_data.loc[1,:], yokoi_data.loc[1,:]])

m_samples_kazuki = kazuki_data.shape[0]
m_samples_non_cancer = non_cancer_data.shape[0]
m_samples_cancer = cancer_data.shape[0]
n_mirnas = non_cancer_data.shape[1] #gets number of columns
miRNA_names_list = list(non_cancer_data.columns)

print("non_cancer_data")
print(non_cancer_data.head)
print("cancer_data")
print(cancer_data.head)

non_cancer_data
<bound method NDFrame.head of     MIMAT0000062  MIMAT0000063  MIMAT0000064  MIMAT0000065  MIMAT0000066  \
0         -3.790         1.143        -3.790        -3.790        -3.790   
0          2.219         1.248        -4.493        -0.432         0.714   
0         -3.950        -3.950        -3.950        -3.950        -3.950   
0         -3.891        -3.891        -3.891        -3.891         1.657   
0          1.479         2.402         1.741         1.132         2.611   
..           ...           ...           ...           ...           ...   
0          4.679         5.000         2.566         4.173         3.739   
0         -0.275         4.578         4.967         5.015         3.895   
0          2.362         3.060         3.977         0.073        -1.200   
0          3.270         3.645         2.707         4.829        -1.056   
0          5.893         4.992         5.696         4.141         3.940   

    MIMAT0000067  MIMAT0000068  MIMAT0000

In [16]:
W = kazuki_data.values
print(W.shape)

(4965, 2565)


In [ ]:
CONST_k = 0.95
significant_pairs = np.zeros((n_mirnas, n_mirnas))
for i in range(0,n_mirnas):
    W_current_miRNA = W[:, i:i+1] #shape (m_samples, 1)
    W_rest_miRNA = W[:, i+1:] #shape (m_samples, n_mirnas-i-1)
    W_comparisons = W_rest_miRNA < W_current_miRNA
    W_greater = np.sum(W_comparisons, axis=0) / m_samples_non_cancer
    W_mask_greater_correlation = np.add((W_greater >= CONST_k).nonzero(), i) #-1
    W_mask_less_correlation = np.add((W_greater <= (1-CONST_k)).nonzero(), i) #-1
    significant_pairs[i][[W_mask_greater_correlation]] = 1
    significant_pairs[i][[W_mask_less_correlation]] = -1


In [52]:
testindicies1 = significant_pairs.nonzero()
print(len(testindicies1[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies1[0])):
    if(significant_pairs[testindicies1[0][i]][testindicies1[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": 1.0")
    else:
        no_greaterthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

1746146
number of Ea<Eb with a<b :972906
number of Ea>Eb with a<b:773240


In [ ]:
Y = cancer_data.values

In [61]:
CONST_l = 0.95
cancer_pairs = np.zeros((n_mirnas, n_mirnas))
diagnostic_pairs = np.zeros((n_mirnas, n_mirnas))
for i in range(0,n_mirnas):
    Y_current_miRNA = Y[:, i:i+1] #shape (m_samples, 1)
    Y_rest_miRNA = Y[:, i+1:] #shape (m_samples, n_mirnas-i-1)
    Y_comparisons = Y_rest_miRNA < Y_current_miRNA
    Y_greater = np.sum(Y_comparisons, axis=0) / m_samples_cancer
    Y_mask_greater_correlation = np.add((Y_greater >= CONST_l).nonzero(), i) #-1
    Y_mask_less_correlation = np.add((Y_greater <= (1-CONST_l)).nonzero(), i) #-1
    cancer_pairs[i][[Y_mask_greater_correlation]] = 1
    cancer_pairs[i][[Y_mask_less_correlation]] = -1
    
    #diagnostic pairs are 
    # significant[mirna1][mirna2] == 1 and cancer[mirna1][mirna2] == -1
    # or significant[mirna][mirna2] == -1 and cancer[mirna1][mirna2] == 1
    #mirna1 is fixed at i
mask1 = np.logical_and(cancer_pairs==1,significant_pairs==-1)
mask2 = np.logical_and(cancer_pairs==-1,significant_pairs==1)
diagnostic_pairs = np.logical_or(mask1, mask2)



In [59]:
testindicies2 = cancer_pairs.nonzero()
print(len(testindicies2[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies2[0])):
    if(significant_pairs[testindicies2[0][i]][testindicies2[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies2[0][i])+", " + str(testindicies2[1][i]) +": 1.0")
    else:
        no_greaterthantest+=1
        #print(str(testindicies2[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

3288330
number of Ea<Eb with a<b :972906
number of Ea>Eb with a<b:2315424


In [62]:
testindicies3 = diagnostic_pairs.nonzero()
print(len(testindicies3[0]))
no_lessthantest = 0
no_greaterthantest = 0
for i in range(len(testindicies3[0])):
    if(significant_pairs[testindicies3[0][i]][testindicies3[1][i]] == 1):
        no_lessthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": 1.0")
    else:
        no_greaterthantest+=1
        #print(str(testindicies[0][i])+", " + str(testindicies[1][i]) +": -1.0")
print("number of Ea<Eb with a<b :" + str(no_lessthantest))
print("number of Ea>Eb with a<b:" +  str(no_greaterthantest))

735
number of Ea<Eb with a<b :735
number of Ea>Eb with a<b:0
